In [1]:
import os
import json
import zlib

from pyflink.common import Configuration, WatermarkStrategy, Row
from pyflink.common.typeinfo import Types
from pyflink.common.serialization import SimpleStringSchema

from pyflink.datastream.state import ValueStateDescriptor
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.functions import FilterFunction
from pyflink.datastream.connectors.kafka import KafkaSource, KafkaOffsetsInitializer

from pyflink.table import StreamTableEnvironment

In [2]:
config = Configuration()
config.set_string("jobmanager.rpc.address", "jobmanager")
config.set_string("rest.address", "jobmanager")
config.set_string("rest.port", "8081")
config.set_string("execution.target", "remote")
config.set_string("table.exec.sink.type", "unified")

env = StreamExecutionEnvironment.get_execution_environment(config)
env.set_parallelism(5)

t_env = StreamTableEnvironment.create(env)

In [3]:
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_password = os.getenv("POSTGRES_PASSWORD", "postgres")
pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = str(os.getenv("POSTGRES_PORT", "5432"))
pg_db = "postgres"

jdbc_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"

In [4]:
def register_sink(table_name, schema):
    t_env.execute_sql(f"""
        CREATE TABLE {table_name} ({schema})
        WITH (
            'connector' = 'jdbc',
            'url' = '{jdbc_url}',
            'table-name' = '{table_name}',
            'username' = '{pg_user}',
            'password' = '{pg_password}',
            'sink.max-retries' = '5',
            'sink.buffer-flush.interval' = '2s'
        )
    """)

In [5]:
register_sink('dim_countries', 'id INT PRIMARY KEY NOT ENFORCED, name STRING')
register_sink('dim_pet_categories', 'id INT PRIMARY KEY NOT ENFORCED, category STRING')
register_sink('dim_customers', 'id INT PRIMARY KEY NOT ENFORCED, customer_first_name STRING, customer_last_name STRING, customer_age INT, customer_email STRING, customer_postal_code STRING, customer_pet_type STRING, customer_pet_name STRING, customer_pet_breed STRING, pet_category STRING, pet_category_id INT, country_id INT')
register_sink('dim_suppliers', 'id INT PRIMARY KEY NOT ENFORCED, supplier_name STRING, supplier_contact STRING, supplier_email STRING, supplier_phone STRING, supplier_address STRING, supplier_city STRING, country_id INT')
register_sink('dim_stores', 'id INT PRIMARY KEY NOT ENFORCED, store_name STRING, store_location STRING, store_city STRING, store_state STRING, store_phone STRING, store_email STRING, country_id INT')
register_sink('dim_sellers', 'id INT PRIMARY KEY NOT ENFORCED, seller_first_name STRING, seller_last_name STRING, seller_email STRING, seller_postal_code STRING, seller_store_id INT, country_id INT')
register_sink('dim_products', 'id INT PRIMARY KEY NOT ENFORCED, product_name STRING, product_category STRING, product_price FLOAT, product_quantity INT, product_weight FLOAT, product_color STRING, product_size STRING, product_brand STRING, product_material STRING, product_description STRING, product_rating FLOAT, product_reviews INT, product_release_date STRING, product_expiry_date STRING, product_supplier_id INT')
register_sink('fact_sales', 'id INT PRIMARY KEY NOT ENFORCED, sale_date STRING, sale_customer_id INT, sale_seller_id INT, sale_product_id INT, sale_store_id INT, sale_supplier_id INT, sale_quantity INT, sale_total_price FLOAT')

In [6]:
source = KafkaSource.builder() \
    .set_bootstrap_servers("kafka:9092") \
    .set_topics("data-topic") \
    .set_group_id("flink_consumer") \
    .set_starting_offsets(KafkaOffsetsInitializer.earliest()) \
    .set_value_only_deserializer(SimpleStringSchema()) \
    .build()

raw_stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "Kafka Source")

parsed_stream = raw_stream \
    .map(lambda msg: json.loads(msg) if msg else None, output_type=Types.PICKLED_BYTE_ARRAY()) \
    .filter(lambda x: x is not None)

In [7]:
class DeduplicateFilter(FilterFunction):
    def open(self, runtime_context):
        self.seen_state = runtime_context.get_state(ValueStateDescriptor("seen", Types.BOOLEAN()))

    def filter(self, value):
        if self.seen_state.value():
            return False
        self.seen_state.update(True)
        return True

def gen_id(val):
    if val is None: return 0
    return zlib.crc32(str(val).encode('utf-8')) & 0x7FFFFFFF

def extract_countries(record):
    return [record[c] for c in['customer_country', 'seller_country', 'store_country', 'supplier_country'] if record.get(c)]

In [8]:
countries_ds = parsed_stream \
    .flat_map(extract_countries, output_type=Types.STRING()) \
    .key_by(lambda x: x) \
    .filter(DeduplicateFilter()) \
    .map(lambda c: Row(gen_id(c), c), output_type=Types.ROW([Types.INT(), Types.STRING()]))

pet_cats_ds = parsed_stream \
    .flat_map(lambda r: [r.get('pet_category')] if r.get('pet_category') else[], output_type=Types.STRING()) \
    .key_by(lambda x: x) \
    .filter(DeduplicateFilter()) \
    .map(lambda c: Row(gen_id(c), c), output_type=Types.ROW([Types.INT(), Types.STRING()]))

customers_ds = parsed_stream.map(lambda r: Row(
    int(r.get('id', 0)), r.get('customer_first_name'), r.get('customer_last_name'),
    r.get('customer_age'), r.get('customer_email'), r.get('customer_postal_code'),
    r.get('customer_pet_type'), r.get('customer_pet_name'), r.get('customer_pet_breed'),
    r.get('pet_category'), gen_id(r.get('pet_category')), gen_id(r.get('customer_country'))
), output_type=Types.ROW([Types.INT(), Types.STRING(), Types.STRING(), Types.INT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.INT(), Types.INT()]))

stores_ds = parsed_stream.map(lambda r: Row(
    int(r.get('id', 0)), r.get('store_name'), r.get('store_location'), r.get('store_city'),
    r.get('store_state'), r.get('store_phone'), r.get('store_email'), gen_id(r.get('store_country'))
), output_type=Types.ROW([Types.INT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.INT()]))

suppliers_ds = parsed_stream.map(lambda r: Row(
    int(r.get('id', 0)), r.get('supplier_name'), r.get('supplier_contact'), r.get('supplier_email'),
    r.get('supplier_phone'), r.get('supplier_address'), r.get('supplier_city'), gen_id(r.get('supplier_country'))
), output_type=Types.ROW([Types.INT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.INT()]))

sellers_ds = parsed_stream.map(lambda r: Row(
    int(r.get('id', 0)), r.get('seller_first_name'), r.get('seller_last_name'), r.get('seller_email'),
    r.get('seller_postal_code'), int(r.get('sale_store_id', 0)), gen_id(r.get('seller_country'))
), output_type=Types.ROW([Types.INT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.INT(), Types.INT()]))

products_ds = parsed_stream.map(lambda r: Row(
    int(r.get('id', 0)), r.get('product_name'), r.get('product_category'), float(r.get('product_price', 0.0)),
    r.get('product_quantity'), float(r.get('product_weight', 0.0)), r.get('product_color'), r.get('product_size'),
    r.get('product_brand'), r.get('product_material'), r.get('product_description'), float(r.get('product_rating', 0.0)),
    r.get('product_reviews'), r.get('product_release_date'), r.get('product_expiry_date'), int(r.get('sale_supplier_id', 0))
), output_type=Types.ROW([Types.INT(), Types.STRING(), Types.STRING(), Types.FLOAT(), Types.INT(), Types.FLOAT(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.STRING(), Types.FLOAT(), Types.INT(), Types.STRING(), Types.STRING(), Types.INT()]))

sales_ds = parsed_stream.map(lambda r: Row(
    int(r.get('id', 0)), r.get('sale_date'), int(r.get('sale_customer_id', 0)), int(r.get('sale_seller_id', 0)),
    int(r.get('sale_product_id', 0)), int(r.get('sale_store_id', 0)), int(r.get('sale_supplier_id', 0)),
    r.get('sale_quantity'), float(r.get('sale_total_price', 0.0))
), output_type=Types.ROW([Types.INT(), Types.STRING(), Types.INT(), Types.INT(), Types.INT(), Types.INT(), Types.INT(), Types.INT(), Types.FLOAT()]))

In [9]:
statement_set = t_env.create_statement_set()

statement_set.add_insert("dim_countries", t_env.from_data_stream(countries_ds).alias("id", "name"))
statement_set.add_insert("dim_pet_categories", t_env.from_data_stream(pet_cats_ds).alias("id", "category"))

statement_set.add_insert("dim_customers", t_env.from_data_stream(customers_ds).alias(
    "id", "customer_first_name", "customer_last_name", "customer_age", "customer_email",
    "customer_postal_code", "customer_pet_type", "customer_pet_name", "customer_pet_breed",
    "pet_category", "pet_category_id", "country_id"))

statement_set.add_insert("dim_stores", t_env.from_data_stream(stores_ds).alias(
    "id", "store_name", "store_location", "store_city", "store_state", "store_phone", "store_email", "country_id"))

statement_set.add_insert("dim_suppliers", t_env.from_data_stream(suppliers_ds).alias(
    "id", "supplier_name", "supplier_contact", "supplier_email", "supplier_phone", "supplier_address", "supplier_city", "country_id"))

statement_set.add_insert("dim_sellers", t_env.from_data_stream(sellers_ds).alias(
    "id", "seller_first_name", "seller_last_name", "seller_email", "seller_postal_code", "seller_store_id", "country_id"))

statement_set.add_insert("dim_products", t_env.from_data_stream(products_ds).alias(
    "id", "product_name", "product_category", "product_price", "product_quantity", "product_weight", "product_color", "product_size",
    "product_brand", "product_material", "product_description", "product_rating", "product_reviews", "product_release_date", "product_expiry_date", "product_supplier_id"))

statement_set.add_insert("fact_sales", t_env.from_data_stream(sales_ds).alias(
    "id", "sale_date", "sale_customer_id", "sale_seller_id", "sale_product_id", "sale_store_id", "sale_supplier_id", "sale_quantity", "sale_total_price"))

In [10]:
statement_set.execute()